# EDA on the VM: channels, ball gaps, long-gap samples

**Run this notebook on the VM only.** Once it has run, its outputs contain competition data:

- never commit it with outputs, and never copy it back to a laptop or paste it into a hosted AI;
- edit the clean copy in the team repo and `scp` it to the VM;
- if a copy with outputs ever has to move, clear it first:
  `jupyter nbconvert --clear-output --inplace eda_vm.ipynb`.

The script commands are **comments**. Run them in a VM terminal (or remove the `#` to run them
from here with `!`). The cells below each command only read what the scripts wrote under
`EDA_OUT` (default `~/eda_out`) and plot it.

Data paths come from `games.asset_paths()`, which honours `MTAH_DATA_DIR`. The public repo comes
from `MTAH_PUBLIC_REPO` (default `~/more-than-a-hack-2027`). Set both in the shell that starts
Jupyter; a kernel started some other way may not see your `~/.bashrc` exports.

In [ ]:
import os
import sys
from pathlib import Path

PUBLIC_REPO = Path(os.environ.get("MTAH_PUBLIC_REPO", Path.home() / "more-than-a-hack-2027"))
EDA_OUT = Path(os.environ.get("EDA_OUT", Path.home() / "eda_out"))
os.environ["MTAH_PUBLIC_REPO"] = str(PUBLIC_REPO)   # for the commented ! commands

sys.path.insert(0, str(PUBLIC_REPO))
import games  # noqa: E402

print("public repo :", PUBLIC_REPO)
print("MTAH_DATA_DIR:", os.environ.get("MTAH_DATA_DIR", "(not set: games.py default)"))
print("EDA_OUT     :", EDA_OUT)
for name, path in games.asset_paths().items():
    print("  {:<14} {}".format(name, "ok" if Path(path).is_file() else "MISSING"))
print("graded blocks:", games.half_video_blocks("BAR-ATH"))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

BINS = ["<0.5s", "0.5-2s", "2-5s", ">=5s"]
SCOPES = ["overall", "half 1", "half 2"]
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def load_summary(folder):
    """ball_gaps_summary.csv from EDA_OUT/folder, or None."""
    path = EDA_OUT / folder / "ball_gaps_summary.csv"
    if not path.is_file():
        print("not found:", path)
        return None
    df = pd.read_csv(path, keep_default_na=False, dtype=str)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df


def get(df, analysis, scope, bin_, metric):
    sel = df[(df.analysis == analysis) & (df.scope == scope) & (df.bin == bin_) &
             (df.metric == metric)]["value"]
    return float(sel.iloc[0]) if len(sel) else np.nan

## 1. `check_channels.py`: what the ATD metadata declares

Is there a ball-status or ball-owner channel, and how is each frame laid out?

In [ ]:
# cd ~/eda && PYTHONPATH="${MTAH_PUBLIC_REPO:-$HOME/more-than-a-hack-2027}" python3 check_channels.py "$HOME/eda_out/channels"
# cd ~/eda && PYTHONPATH="${MTAH_PUBLIC_REPO:-$HOME/more-than-a-hack-2027}" python3 check_channels.py "$HOME/eda_out/channels_broadcast" --source broadcast

In [ ]:
def parse_channels(path):
    """(summary text, [(spec, first, last)], {spec: [(block label, n names)]})."""
    summary, _, full = path.read_text().partition("\n\n")
    specs, blocks, spec = [], {}, None
    for line in full.splitlines():
        if line.startswith("# spec "):
            spec = int(line.split()[2].rstrip(":"))
            first, last = line.split("frames ")[1].split("..")
            specs.append((spec, first, last))
            blocks[spec] = []
        elif spec is None:
            continue
        elif line.startswith("## block"):
            blocks[spec].append([line.split(":", 1)[1].strip(), 0])
        elif line.strip() and blocks[spec]:
            blocks[spec][-1][1] += 1
    return summary, specs, blocks


for folder in ["channels", "channels_broadcast"]:
    path = EDA_OUT / folder / "channels.txt"
    if not path.is_file():
        print("not found:", path)
        continue
    summary, specs, blocks = parse_channels(path)
    print("==", folder, "==")
    print(summary)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5),
                                   gridspec_kw={"width_ratios": [1.2, 1]})
    labels = [b[0] if len(b[0]) <= 45 else b[0][:42] + "..." for b in blocks[specs[0][0]]]
    counts = [b[1] for b in blocks[specs[0][0]]]
    ax1.barh(range(len(counts)), counts, color="#4C72B0")
    ax1.set_yticks(range(len(counts)), labels, fontsize=8)
    ax1.invert_yaxis()
    ax1.set_xlabel("declared channel names")
    ax1.set_title("Frame layout, spec 1 (in raw-line order)")
    for i, c in enumerate(counts):
        ax1.text(c, i, " " + str(c), va="center", fontsize=8)

    for spec, first, last in specs:
        try:
            a, b = int(first), int(last)
        except ValueError:
            continue
        ax2.barh(spec, b - a + 1, left=a, color="#55A868")
    ax2.set_yticks([s for s, _, _ in specs])
    ax2.invert_yaxis()
    ax2.set_xlabel("ATD frame_id")
    ax2.set_ylabel("spec")
    status = "status/owner channel: " + ("NONE" if "NONE declared" in summary else "candidates found")
    ax2.set_title("{} data-format spec(s); {}".format(len(specs), status))
    fig.suptitle(folder)
    fig.tight_layout()
    plt.show()

## 2. `ball_gaps.py`: where the ball goes missing, and what happens across a gap

The main run, plus two sensitivity runs (set-piece tolerance of ±1 s, flip distance cap of 5 m).

In [ ]:
# cd ~/eda && PYTHONPATH="${MTAH_PUBLIC_REPO:-$HOME/more-than-a-hack-2027}" python3 ball_gaps.py "$HOME/eda_out/ball_gaps"
# cd ~/eda && PYTHONPATH="${MTAH_PUBLIC_REPO:-$HOME/more-than-a-hack-2027}" python3 ball_gaps.py "$HOME/eda_out/ball_gaps_tol1s" --tag-tolerance 1
# cd ~/eda && PYTHONPATH="${MTAH_PUBLIC_REPO:-$HOME/more-than-a-hack-2027}" python3 ball_gaps.py "$HOME/eda_out/ball_gaps_maxdist5" --flip-max-dist 5

In [ ]:
RUNS = {"main": "ball_gaps", "tag ±1 s": "ball_gaps_tol1s", "flip ≤ 5 m": "ball_gaps_maxdist5"}
summaries = {name: load_summary(folder) for name, folder in RUNS.items()}
bg_main = summaries["main"]

### (a) Gap lengths: many short gaps, or a few long ones holding most missing frames?

In [ ]:
df = bg_main
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))
x = np.arange(len(BINS))
w = 0.27
for i, scope in enumerate(SCOPES):
    n = [get(df, "gap_length", scope, b, "n_gaps") for b in BINS]
    share = [get(df, "gap_length", scope, b, "share_of_missing_frames") for b in BINS]
    ax1.bar(x + (i - 1) * w, n, w, label=scope)
    ax2.bar(x + (i - 1) * w, share, w, label=scope)
for ax, ylab in ((ax1, "number of gaps"), (ax2, "% of missing frames")):
    ax.set_xticks(x, BINS)
    ax.set_xlabel("gap length")
    ax.set_ylabel(ylab)
ax1.legend()
ax1.set_title("Gap count per length bin")
ax2.set_title("Where the missing frames sit")
fig.suptitle("Ball missing on {:.1f}% of {:,.0f} graded frames; {:.0f} gaps, {:.0f} touch a "
             "half edge".format(get(df, "gap_length", "overall", "all", "missing_share_of_graded"),
                                get(df, "gap_length", "overall", "all", "graded_frames"),
                                get(df, "gap_length", "overall", "all", "n_gaps"),
                                get(df, "gap_length", "overall", "all", "edge_gaps")))
fig.tight_layout()
plt.show()

### (b) + (e) Long gaps vs SET PIECES: is the real overlap above chance?

Dot = real share of gaps overlapping a SET PIECES tag. Bar = null p5–p95 from circularly shifted
tags, with a tick at the null mean. Real above the band supports "long gap = stoppage".

In [ ]:
labels = [">=long", "2-5s", ">=5s"]
fig, ax = plt.subplots(figsize=(8, 3.8))
plotted = False
for k, (name, off) in enumerate((("main", -0.12), ("tag ±1 s", 0.12))):
    df = summaries[name]
    if df is None or not (df.analysis == "set_pieces_null").any():
        print(name, ": set-piece null not in the summary (no Smart Tagging file, or skipped)")
        continue
    plotted = True
    color = ["#4C72B0", "#DD8452"][k]
    for i, lab in enumerate(labels):
        lo, hi = (get(df, "set_pieces_null", "overall", lab, m) for m in ("null_p5", "null_p95"))
        mean = get(df, "set_pieces_null", "overall", lab, "null_mean")
        real = get(df, "set_pieces_null", "overall", lab, "real_share")
        p = get(df, "set_pieces_null", "overall", lab, "p_emp")
        n = get(df, "set_pieces_null", "overall", lab, "n_gaps")
        if np.isnan(real):
            continue
        ax.vlines(i + off, lo, hi, color=color, alpha=0.35, lw=10,
                  label=name + " null p5–p95" if i == 0 else None)
        ax.hlines(mean, i + off - 0.06, i + off + 0.06, color=color)
        ax.plot(i + off, real, "o", color=color, ms=8, label=name + " real" if i == 0 else None)
        ax.annotate("p={:.3f}\nn={:.0f}".format(p, n), (i + off, real), xytext=(8, 0),
                    textcoords="offset points", fontsize=7, va="center")
if plotted:
    ax.set_xticks(range(len(labels)), ["≥ long-gap threshold", "2-5 s", "≥ 5 s"])
    ax.set_ylabel("% of gaps overlapping SET PIECES")
    ax.set_title("Real overlap vs circular-shift null")
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    plt.show()
else:
    plt.close(fig)

### (c) Crowding: is the ball lost in contested play?

In [ ]:
df = bg_main
groups = [("gap start\n(all)", "gap_start_all"), ("gap start\n≥ long", "gap_start_long"),
          ("gap start\n< long", "gap_start_short"), ("ball visible\n(baseline)", "visible")]
means = [get(df, "crowding", s, "", "mean") for _, s in groups]
medians = [get(df, "crowding", s, "", "median") for _, s in groups]
ns = [get(df, "crowding", s, "", "n") for _, s in groups]
radius = get(df, "params", "all", "", "radius")

fig, ax = plt.subplots(figsize=(8, 3.8))
colors = ["#C44E52"] * 3 + ["#8C8C8C"]
ax.bar(range(len(groups)), means, color=colors, label="mean")
ax.plot(range(len(groups)), medians, "k_", ms=25, mew=2, label="median")
top = np.nanmax(means + medians)
for i, (m, med, n) in enumerate(zip(means, medians, ns)):
    ax.text(i, np.nanmax([m, med]) + 0.03 * top, "mean {:.2f}\nn={:.0f}".format(m, n),
            ha="center", va="bottom", fontsize=8)
ax.set_ylim(0, top * 1.35)
ax.set_xticks(range(len(groups)), [g for g, _ in groups])
ax.set_ylabel("players within {:g} m of the ball".format(radius))
ax.set_title("Crowding at the last-seen ball vs ball-visible frames")
ax.legend(loc="upper left", ncol=2, fontsize=8)
fig.tight_layout()
plt.show()

### (d) Flips: does the nearest team change across a gap more than it would anyway?

Bars = flip rate across gaps. Grey = reference rate on ball-visible frame pairs the same distance
apart. Above each bar: flips / evaluable gaps.

In [ ]:
panels = [(n, d) for n, d in (("main", summaries["main"]), ("flip ≤ 5 m", summaries["flip ≤ 5 m"]))
          if d is not None and (d.analysis == "flip").any()]
if not panels:
    print("no flip rows (run without --meta?)")
else:
    fig, axes = plt.subplots(1, len(panels), figsize=(6 * len(panels), 3.8), sharey=True,
                             squeeze=False)
    x = np.arange(len(BINS))
    for ax, (name, df) in zip(axes[0], panels):
        rate = [get(df, "flip", "overall", b, "flip_rate") for b in BINS]
        ref = [get(df, "flip", "overall", b, "ref_flip_rate") for b in BINS]
        ax.bar(x - 0.2, rate, 0.4, color="#4C72B0", label="across gaps")
        ax.bar(x + 0.2, ref, 0.4, color="#8C8C8C", label="reference pairs")
        for i, b in enumerate(BINS):
            k = get(df, "flip", "overall", b, "n_flip")
            n = get(df, "flip", "overall", b, "n_evaluable")
            ax.text(i - 0.2, 0 if np.isnan(rate[i]) else rate[i], "{:.0f}/{:.0f}".format(k, n),
                    ha="center", va="bottom", fontsize=8)
        allc = [get(df, "flip", "overall", "all", m) for m in ("n_flip", "n_evaluable", "n_gaps")]
        ax.set_xticks(x, BINS)
        ax.set_xlabel("gap length")
        ax.set_title("{}: overall {:.0f}/{:.0f} evaluable of {:.0f} gaps".format(name, *allc))
        ax.legend(fontsize=8, loc="upper center", ncol=2)
    top = max(np.nanmax([get(d, "flip", "overall", b, m) for b in BINS
                         for m in ("flip_rate", "ref_flip_rate")] + [0.05]) for _, d in panels)
    axes[0][0].set_ylim(0, top * 1.35)
    axes[0][0].set_ylabel("share with nearest-team change")
    fig.tight_layout()
    plt.show()

    df = panels[0][1]
    reasons = ["no_last_seen", "no_reappearance", "no_player", "too_far"]
    counts = {r: get(df, "flip", "overall", "all", "not_eval_" + r) for r in reasons}
    print("not evaluable ({}):".format(panels[0][0]),
          ", ".join("{} {:.0f}".format(r, c) for r, c in counts.items()))

## 3. `sample_long_gaps.py`: gaps to watch by hand

Five length strata, half-open in seconds: [0, 5), [5, 10), [10, 20), [20, 35), [35, ∞).
Each file holds up to 8 gaps, spread over both halves, excluding the public GT window and the
second-half frames before kickoff. These tables are **frame-indexed**: fill in `category` and
`notes` while watching, and keep the files on the VM.

In [ ]:
# cd ~/eda && PYTHONPATH="${MTAH_PUBLIC_REPO:-$HOME/more-than-a-hack-2027}" python3 sample_long_gaps.py "$HOME/eda_out/long_gaps" --min-s 0  --max-s 5  --csv-name gap_00_05_sample.csv
# cd ~/eda && PYTHONPATH="${MTAH_PUBLIC_REPO:-$HOME/more-than-a-hack-2027}" python3 sample_long_gaps.py "$HOME/eda_out/long_gaps" --min-s 5  --max-s 10 --csv-name gap_05_10_sample.csv
# cd ~/eda && PYTHONPATH="${MTAH_PUBLIC_REPO:-$HOME/more-than-a-hack-2027}" python3 sample_long_gaps.py "$HOME/eda_out/long_gaps" --min-s 10 --max-s 20 --csv-name gap_10_20_sample.csv
# cd ~/eda && PYTHONPATH="${MTAH_PUBLIC_REPO:-$HOME/more-than-a-hack-2027}" python3 sample_long_gaps.py "$HOME/eda_out/long_gaps" --min-s 20 --max-s 35 --csv-name gap_20_35_sample.csv
# cd ~/eda && PYTHONPATH="${MTAH_PUBLIC_REPO:-$HOME/more-than-a-hack-2027}" python3 sample_long_gaps.py "$HOME/eda_out/long_gaps" --min-s 35            --csv-name gap_35plus_sample.csv

In [ ]:
def show_sample(name):
    path = EDA_OUT / "long_gaps" / name
    if not path.is_file():
        print("not found:", path)
        return
    table = pd.read_csv(path, keep_default_na=False)
    print("{}: {} gap(s)".format(name, len(table)))
    display(table)

### 0–5 s

In [ ]:
show_sample("gap_00_05_sample.csv")

### 5–10 s

In [ ]:
show_sample("gap_05_10_sample.csv")

### 10–20 s

In [ ]:
show_sample("gap_10_20_sample.csv")

### 20–35 s

In [ ]:
show_sample("gap_20_35_sample.csv")

### 35 s and longer

In [ ]:
show_sample("gap_35plus_sample.csv")